In [ ]:
# Path configuration
from pathlib import Path

PROJECT_ROOT = Path("..")  # Repository root.
BASELINE = PROJECT_ROOT / "data"
FIRM_DATA = PROJECT_ROOT
PATSTAT = PROJECT_ROOT / "patstat"
PATENT_TEXT = PROJECT_ROOT / "patent_text"
SAMPLEDATA = PROJECT_ROOT / "sampledata"
DATA = Path("data")          # Generated revision intermediates.
OUTPUTS = Path("outputs")    # Tables, figures and saved comparison results.
SEED = 42


# CPC corpora and CPC2vec training

Train detailed and main-group CPC2vec models for GBB identification and the SI S3.6 comparison. Run from `dataprep/` with paths set in the first cell. Both corpora come from the same PATSTAT family/CPC dataframe; each patent family forms a document.

Retain families with more than one code. Write one sentence for lists of at most six codes and ten sampled six-code sentences for longer lists. Train 50-dimensional skip-gram models on the resulting corpora.


In [ ]:
import csv
import numpy as np
import pandas as pd

CORPUS = DATA / 'upstream/cpc2vec'
MODEL_DIR = BASELINE / 'external'
CORPUS.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DETAILED_INPUT = PATSTAT / 'tls225.parquet'
DETAILED_CORPUS = CORPUS / 'cpc_detailed.txt'
MAIN_GROUP_CORPUS = CORPUS / 'cpc_main_group.txt'
DETAILED_MODEL = MODEL_DIR / 'cpc_sg_ns_dim50.model'
MAIN_GROUP_MODEL = MODEL_DIR / 'cpc8d_noy02_sg_ns_dim50.model'


## Detailed CPC lists by patent family

Read `docdb_family_id,cpc_class_symbol`. Deduplicate family/code rows, remove spaces from symbols, count memberships, and retain families with more than one code. Shuffle the rows and group the codes by family.


In [ ]:
detailed = pd.read_parquet(
    DETAILED_INPUT, columns=['docdb_family_id', 'cpc_class_symbol']
).rename(columns={'cpc_class_symbol': 'cpc'}).drop_duplicates()
detailed['cpc'] = detailed.cpc.astype(str).str.replace(' ', '', regex=False)
detailed['num_cpc'] = detailed.groupby('docdb_family_id').cpc.transform('count')
detailed = detailed[detailed.num_cpc > 1].sample(frac=1, random_state=SEED)
detailed_lists = detailed.groupby('docdb_family_id', sort=False).cpc.agg(list)
detailed_lists.head()


## Write the short training sentences

Write each list of at most six codes once. For longer lists, make ten passes and sample six positions without replacement in each pass. Combine the short lists and sampled sentences in each output corpus.


In [ ]:
def write_corpus(code_lists, path, seed):
    sizes = code_lists.map(len)
    small = code_lists[sizes <= 6]
    large = code_lists[sizes > 6]
    rng = np.random.RandomState(seed)
    with path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.writer(handle, delimiter=' ', lineterminator='\n')
        writer.writerows(small)
        for repeat in range(10):
            for codes in large:
                writer.writerow(rng.choice(codes, size=6, replace=False))
    return {'documents': len(code_lists),
            'sentences': len(small) + 10 * len(large),
            'tokens': int(sizes[sizes <= 6].sum()) + 60 * len(large)}


detailed_counts = write_corpus(detailed_lists, DETAILED_CORPUS, SEED)
del detailed_lists
pd.Series(detailed_counts)


## Fit the detailed model

Train a 50-dimensional skip-gram model with the settings below. Save its vectors and vocabulary together. Cosine calculations normalize these vectors at the analysis stage.


In [ ]:
from gensim.models import Word2Vec
from gensim.models.word2vec import LineSentence

settings = dict(vector_size=50, window=6, min_count=1, workers=8,
                sg=1, negative=5, hs=0, epochs=10, seed=1)
detailed_model = Word2Vec(LineSentence(str(DETAILED_CORPUS)), **settings)
detailed_model.save(str(DETAILED_MODEL))
pd.Series({'vocabulary': len(detailed_model.wv),
           'dimensions': detailed_model.vector_size})


## Main-group lists from the detailed dataframe

Start from `detailed` above. Collapse each code to the text before `/`, deduplicate family/main-group pairs, and count distinct main groups per family. Retain families with more than one main group and use the same sentence-sampling procedure.


In [ ]:
main_group = detailed[['docdb_family_id', 'cpc']].copy()
main_group['cpc'] = main_group.cpc.str.split('/').str[0]
main_group = main_group.drop_duplicates()
main_group['num_cpc'] = main_group.groupby('docdb_family_id').cpc.transform('count')
main_group = main_group[main_group.num_cpc > 1].sample(frac=1, random_state=SEED)
main_group_lists = main_group.groupby('docdb_family_id', sort=False).cpc.agg(list)
main_group_lists.head()


In [ ]:
main_group_counts = write_corpus(main_group_lists, MAIN_GROUP_CORPUS, SEED)
del detailed, main_group, main_group_lists
main_group_model = Word2Vec(LineSentence(str(MAIN_GROUP_CORPUS)), **settings)
main_group_model.save(str(MAIN_GROUP_MODEL))
pd.DataFrame([
    {'model': 'detailed', **detailed_counts, 'vocabulary': len(detailed_model.wv)},
    {'model': 'main_group', **main_group_counts,
     'vocabulary': len(main_group_model.wv)},
])


## Downstream use

Save both models under `BASELINE/external/`. The detailed model is read by `../2_cpc.ipynb`, the clustering notebooks and revision notebooks 04–08. Revision notebook 07 reads the main-group model. Corpus files are saved under `DATA/upstream/cpc2vec/`.
